#Department Table Cleaning

##importing dependency and data

In [0]:
import pyspark.sql.functions as F

In [0]:
df_employee_raw = spark.read.table("bronze.azure_blob_storage.employee")

In [0]:
df_employee_raw.display()

##Removing columns inserted by fivetron

In [0]:
df_employee=df_employee_raw.drop('_file','_line','_modified','_fivetran_synced')

In [0]:
df_employee.display()

##Trimming column names


In [0]:
for column in df_employee.columns:
    df_employee=df_employee.withColumn(column,F.trim(F.col(column)))
df_employee.display()

## Type Casting

#### converting to int

In [0]:
df_employee=df_employee.withColumn("employee_id",F.col("employee_id").cast("int"))
df_employee=df_employee.withColumn("department_id",F.col("department_id").cast("int"))
df_employee=df_employee.withColumn("company_id",F.col("company_id").cast("int"))
# df_employee=df_employee.withColumn("base_salary",F.col("base_salary").cast("int"))
df_employee.display()

#### Convert to boolean

In [0]:
df_employee = df_employee.withColumn('is_active', F.when(F.col('is_active') == "true", True).otherwise(False))
df_employee.display()

#### coverting to double

In [0]:
df_employee=df_employee.withColumn("base_salary",F.col("base_salary").cast("double"))
df_employee.display()

#### Date and time handling

In [0]:
df_employee = df_employee.withColumn(
    "split_date_time",
    F.split(F.col("termination_date"), " ")
)
df_employee = df_employee.withColumn(
    "termination_date",
    F.col("split_date_time").getItem(0)
).withColumn(
    "termination_time",
    F.col("split_date_time").getItem(1)
)
df_employee = df_employee.drop("split_date_time")
df_employee.display()

In [0]:
df_employee = df_employee.withColumn(
    "split_date_time",
    F.split(F.col("hire_date"), " ")
)
df_employee = df_employee.withColumn(
    "hire_date",
    F.col("split_date_time").getItem(0)
).withColumn(
    "hire_time",
    F.col("split_date_time").getItem(1)
)
df_employee = df_employee.drop("split_date_time")
df_employee.display()

In [0]:
df_employee = df_employee.drop('hire_time', 'termination_time')

In [0]:
df_employee = df_employee.withColumn(
    "hire_date",
    F.to_date(F.col("hire_date"), "dd-MM-yyyy")
)
df_employee.display()

In [0]:
df_employee = df_employee.withColumn(
    "termination_date",
    F.to_date(F.col("termination_date"), "dd-MM-yyyy")
)
df_employee.display()

In [0]:
df_employee = df_employee.withColumn('hire_date', F.date_format(F.col('hire_date'), 'yyyy-MM-dd'))

In [0]:
df_employee = df_employee.withColumn('termination_date', F.date_format(F.col('termination_date'), 'yyyy-MM-dd'))

In [0]:
df_employee.display()

In [0]:
df_employee = df_employee.withColumn(
    "termination_date",
    F.to_date(F.col("termination_date"), "yyyy-MM-dd")
)
df_employee.display()

In [0]:
df_employee = df_employee.withColumn(
    "hire_date",
    F.to_date(F.col("hire_date"), "yyyy-MM-dd")
)
df_employee.display()

## Handling null values

In [0]:
for column in df_employee.columns:
    null_count = df_employee.filter(F.col(column).isNull()).count()
    print(f"{column}: {null_count}")

In [0]:
df_employee = df_employee.withColumn(
    "termination_date",
    F.when(
        F.col("termination_date").isNull(),
        F.date_format(F.current_date(), "yyyy-MM-dd")
    ).otherwise(F.date_format(F.col("termination_date"), "yyyy-MM-dd"))
)
df_employee.display()

In [0]:
df_employee = df_employee.withColumn(
    "termination_date",
    F.to_date(F.col("termination_date"), "yyyy-MM-dd")
)
df_employee.display()

## Handling Duplicates

In [0]:
df_employee.exceptAll(df_employee.dropDuplicates()).show()

In [0]:
df_employee.display()

In [0]:
df_employee.write.mode("overwrite").saveAsTable("silver.transformation.employee")